In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from feature_engine.encoding import CountFrequencyEncoder
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import shap
import mlflow.sklearn
import mlflow

import config

In [2]:
# define Mlflow env. settings
mlflow.set_tracking_uri(uri=config.mlflow_uri)
%env MLFLOW_TRACKING_URI= config.mlflow_db
%env mlflow_tracking_username = config.mlflow_un
%env mlflow_tracking_password = config.mlflow_pw
mlflow.enable_system_metrics_logging()
mlflow.autolog()

env: MLFLOW_TRACKING_URI=config.mlflow_db
env: mlflow_tracking_username=config.mlflow_un
env: mlflow_tracking_password=config.mlflow_pw


2024/07/04 19:02:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2024/07/04 19:02:26 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


In [3]:
#mlflow.autolog(log_models=True, log_datasets=True)
mlflow.set_experiment("Extended Isolation Forest Experiment")
customerList = ["X5","X7","X8"]#,"X3","X4","X1","X2","X6"] #
with mlflow.start_run(run_name=f"Sklearn_IF_Base_FreqEnc_ColdStart", nested=True) as grandparent_run:
    for customer in customerList:
        filepath = Path(f'C:/MLAR_DATA/T0001/FS_Base_data/FS_Base_data_{customer}.parquet')
        if filepath.is_file()==True:
            df = pd.read_parquet(filepath)
            df = df.drop(['T0083','T0083_SEVERITY'],axis=1) #,'T_MLAR','T_MLAR_SEVERITY','MLAR_TYPE'
            df = df.astype(str).replace('nan',np.nan).dropna(how='all',axis=1)
            features = df.columns.tolist()
            categoricalColumns = df.select_dtypes(include=['object']).columns.tolist()
            encoder = CountFrequencyEncoder(encoding_method='frequency', variables=categoricalColumns, ignore_format=True)
            dataFrame = encoder.fit_transform(df) 

            # Define model parameters
            numTrees = 100
            #sampleSize = 256
            random = np.random.RandomState(1234)

            # Initialize the model
            if_model = IsolationForest(random_state=random)
            
            # Start an MLflow run
            with mlflow.start_run(run_name=f"Predicted_Dataset", nested=True) as parent_run:
                # Log model parameters and train
                #mlflow.log_param("ntrees", numTrees)
                #mlflow.log_param("sample_size", sampleSize)
                if_model.fit(dataFrame) 
                
                # Predict anomalies and generate histogram
                anomalies = if_model.predict(dataFrame)              
                
                # Combine the scores with the original data
                finalDataframe = pd.concat([pd.DataFrame(data=anomalies,columns=['anomaly']),dataFrame],axis=1)
                predictedDF = finalDataframe[finalDataframe['anomaly'] > 0]

                #Export Dataset
                mlflow.log_table(predictedDF,"predictedDF.json")

                with mlflow.start_run(run_name=f"Explaination", nested=True) as child_run:
                    #Train explaination model
                    explainer = shap.Explainer(if_model.predict, dataFrame)
                    shap_values = explainer.shap_values(dataFrame)
                    #mlflow.shap.log_explanation(if_model.predict, dataFrame)
                    
                    #explainOutput = DRF_model.explain(predictedTest)
                    #predictContribution = DRF_model.predict_contributions(predictedTest)
        else:
            print("Error - Feature store file not found. Check directory.")

2024/07/04 19:02:26 WARNING mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics because creating `GPUMonitor` failed with error: Failed to initialize NVML, skip logging GPU metrics: NVML Shared Library Not Found.
2024/07/04 19:02:26 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2024/07/04 19:19:30 WARNING mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics because creating `GPUMonitor` failed with error: Failed to initialize NVML, skip logging GPU metrics: NVML Shared Library Not Found.
2024/07/04 19:19:30 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2024/07/04 19:21:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\DSE\Python\Lib\site-packages\mlflow\types\utils.py:394: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing valu

In [ ]:
shapBarPlot = shap.plots.bar(shap_values)

In [ ]:
shap.plots.initjs()

In [ ]:
shap.plots.force(shap_values)